In [268]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [269]:
df=pd.read_csv(r"C:\Users\Sunday Ogboi\Downloads\Portfolio\Portfolio project 01\retail_store_sales.csv")
df

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,4/8/2024
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,7/23/2023
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,10/5/2022
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,5/7/2022
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,10/2/2022
...,...,...,...,...,...,...,...,...,...,...
12570,TXN_9347481,CUST_18,Patisserie,Item_23_PAT,38.0,4.0,152.0,Credit Card,In-store,9/3/2023
12571,TXN_4009414,CUST_03,Beverages,Item_2_BEV,6.5,9.0,58.5,Cash,Online,8/12/2022
12572,TXN_5306010,CUST_11,Butchers,Item_7_BUT,14.0,10.0,140.0,Cash,Online,8/24/2024
12573,TXN_5167298,CUST_04,Furniture,Item_7_FUR,14.0,6.0,84.0,Cash,Online,12/30/2023


In [270]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    12575 non-null  object 
 1   Customer ID       12575 non-null  object 
 2   Category          12575 non-null  object 
 3   Item              11362 non-null  object 
 4   Price Per Unit    11966 non-null  float64
 5   Quantity          11971 non-null  float64
 6   Total Spent       11971 non-null  float64
 7   Payment Method    12575 non-null  object 
 8   Location          12575 non-null  object 
 9   Transaction Date  12575 non-null  object 
dtypes: float64(3), object(7)
memory usage: 982.6+ KB


In [271]:
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'])

In [272]:
df['Transaction ID'].nunique() 
#each transaction ID is unique for each sale record

12575

In [273]:
df['Category'].unique().tolist()
#There are 8 distinct Categories

['Patisserie',
 'Milk Products',
 'Butchers',
 'Beverages',
 'Food',
 'Furniture',
 'Electric household essentials',
 'Computers and electric accessories']

In [274]:
df['Item'].isnull().any()

np.True_

In [275]:
df['Item'].nunique()

200

In [276]:
df_filtered_by_items = df[df["Item"].notna()]

In [277]:
grouped_item = df_filtered_by_items.groupby(["Category", "Price Per Unit"])["Item"].nunique().reset_index()
grouped_item

,Category,Price Per Unit,Item
0,Beverages,5.0,1
1,Beverages,6.5,1
2,Beverages,8.0,1
3,Beverages,9.5,1
4,Beverages,11.0,1
...,...,...,...
195,Patisserie,35.0,1
196,Patisserie,36.5,1
197,Patisserie,38.0,1
198,Patisserie,39.5,1


In [278]:
grouped_item = grouped_item.rename(columns={"Item": "distinct_item_count"})

In [279]:
result = grouped_item[grouped_item["distinct_item_count"] > 1]
result
#No two items under the same category has the same price

,Category,Price Per Unit,distinct_item_count


In [280]:
price_list = (df_filtered_by_items.groupby(["Category", "Price Per Unit"])["Item"].unique().reset_index())
price_list

,Category,Price Per Unit,Item
0,Beverages,5.0,[Item_1_BEV]
1,Beverages,6.5,[Item_2_BEV]
2,Beverages,8.0,[Item_3_BEV]
3,Beverages,9.5,[Item_4_BEV]
4,Beverages,11.0,[Item_5_BEV]
...,...,...,...
195,Patisserie,35.0,[Item_21_PAT]
196,Patisserie,36.5,[Item_22_PAT]
197,Patisserie,38.0,[Item_23_PAT]
198,Patisserie,39.5,[Item_24_PAT]


In [281]:
df[df[['Total Spent','Quantity']].isna()][['Quantity', 'Total Spent', 'Price Per Unit']].any()
#No N/A price per unit column also has any N/A row with total spent or quantity

Quantity          False
Total Spent       False
Price Per Unit    False
dtype: bool

In [282]:
df['Price Per Unit'] = df['Price Per Unit'].fillna (df['Total Spent']/df['Quantity'])

In [283]:
df = df.merge(price_list, how='left', on=['Category','Price Per Unit'], suffixes=('', '_from_price_list'))

In [284]:
df['Item'] = df['Item'].fillna(df['Item_from_price_list'])

In [285]:
df = df.drop(columns=['Item_from_price_list'])

In [286]:
df['Quantity'] = df['Quantity'].fillna(df['Total Spent']/df['Price Per Unit'])

In [287]:
df['Total Spent'] = df['Total Spent'].fillna(df['Quantity']*df['Price Per Unit'])

In [288]:
df.info()
#Total spent and Quantity still has a significantly high number of null cells.
#I'll be filling the spaces with the average value per customer per item for each column

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    12575 non-null  object        
 1   Customer ID       12575 non-null  object        
 2   Category          12575 non-null  object        
 3   Item              12575 non-null  object        
 4   Price Per Unit    12575 non-null  float64       
 5   Quantity          11971 non-null  float64       
 6   Total Spent       11971 non-null  float64       
 7   Payment Method    12575 non-null  object        
 8   Location          12575 non-null  object        
 9   Transaction Date  12575 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(3), object(6)
memory usage: 982.6+ KB


In [289]:
df['Customer ID'] = df['Customer ID'].apply(lambda x: x.item() if isinstance(x, np.ndarray) and x.size == 1 else (x if isinstance(x, (list, tuple)) and len(x) > 0 else x))
df['Item'] = df['Item'].apply(lambda x: x.item() if isinstance(x, np.ndarray) and x.size == 1 else (x if isinstance(x, (list, tuple)) and len(x) > 0 else x))

In [290]:
df['Quantity'] = (df['Quantity'].fillna(df.groupby(['Customer ID', 'Item'])['Quantity'].transform('mean'))).round(1)
#The raw dataset contained several structural irregularities and missing values that required an advanced programmatic data-cleaning pipeline to ensure downstream analytical integrity.
#To combat this, I ran the code above which clean up and flatten data inside the Customer ID and Item columns by converting accidental array or list containers into simple, scalar values (like regular strings).

In [291]:
df['Total Spent'] = (df['Total Spent'].fillna(df.groupby(['Customer ID', 'Item'])['Total Spent'].transform('mean'))).round(2)

In [292]:
duplicates = df[df.duplicated(keep=False)]
print(duplicates) 
#No duplicate entries

Empty DataFrame
Columns: [Transaction ID, Customer ID, Category, Item, Price Per Unit, Quantity, Total Spent, Payment Method, Location, Transaction Date]
Index: []


In [295]:
df.info()
#The data is almost fully populated leaving only values with no match in average values per customer per item.
#I would not want to distort the data by fully populating it with the average per customer as it may distort the data

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    12575 non-null  object        
 1   Customer ID       12575 non-null  object        
 2   Category          12575 non-null  object        
 3   Item              12575 non-null  object        
 4   Price Per Unit    12575 non-null  float64       
 5   Quantity          12531 non-null  float64       
 6   Total Spent       12531 non-null  float64       
 7   Payment Method    12575 non-null  object        
 8   Location          12575 non-null  object        
 9   Transaction Date  12575 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(3), object(6)
memory usage: 982.6+ KB


In [294]:
df.to_csv(r"C:\Users\Sunday Ogboi\Downloads\Portfolio\Portfolio project 01/python cleaned.csv",index=False)